# Colab: M2 diagnostic batch

**Why this batch exists.** The PINN passes the easy test (case C: 60 wells, monthly readings) but fails the realistic one (30 wells, 4 readings a year). This batch finds out why, and tests the planned fixes. See `reports/m2_synthetic.md`.

**Before you start:**
1. On the laptop, run `python scripts/pack_for_colab.py`, then upload the new `pinn_code.zip` to `MyDrive/pinn/`, replacing the old one. The code has changed.
2. Choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

**Timing.** 9 runs of about 15 minutes each, so about 2.5 hours. Keep the tab open.

**If Colab disconnects:** just do **Run all** again. Runs that already finished are skipped.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pinn"

In [ ]:
import glob
import os
import shutil
import zipfile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/runs/logs", exist_ok=True)
CODE = "/content/pinn"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_ROOT}/pinn_code.zip") as z:
    z.extractall(CODE)
%cd {CODE}

In [ ]:
# PyTorch is already installed on Colab; this adds everything else (about 1 minute).
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ["PINN_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"
!python scripts/check_env.py
!python -m pytest -q -m "not data"

## Build the simulated aquifers (about 30 seconds)

In [ ]:
!python scripts/make_synthetic.py

## Diagnostic runs (most important first)

| Run | Question it answers |
|---|---|
| d1 | gate case with only 4 readings a year: does sparse sampling alone break it? |
| d2 | case A with 60 wells read monthly: does the missing pumping-cluster pattern alone break it? |
| run4 + data weight 10 | with 0.3 m noise, does trusting the data less stop the model chasing noise? |
| run3 + longer frozen stage | does letting the fit settle before S and T move stop S crashing? |
| run3 + report-style prior, 30% off | does a CGWB-style starting value (deliberately wrong) rescue S? |
| run3 + T fixed at 1950 (30% off) | the planned fallback if T can't be learned |
| run3 + data weight 10 | does the lower data weight also help without noise? |
| run4 + data weight 1 | an even lower data weight for noisy data |
| run1, seed 2 | third repeat of the gate (it passed on 1 of 2 seeds) |

In [ ]:
DIAG = [
    ("d1", "", "diag"),
    ("d2", "", "diag"),
    ("run4", "--set training.weighting.init=10", "diag_w10"),
    ("run3", "--set training.stages.physics_frozen=5000", "diag_frozen5k"),
    ("run3", "--prior-factor 1.3", "diag_prior130"),
    ("run3", "--fix-T 1950", "diag_fixT1950"),
    ("run3", "--set training.weighting.init=10", "diag_w10"),
    ("run4", "--set training.weighting.init=1", "diag_w1"),
    ("run1", "--seed 2", "diag_seed2"),
]
for run, extra, tag in DIAG:
    if glob.glob(f"{DRIVE_ROOT}/runs/*_m2_{run}_{tag}/metrics.json"):
        print("already done:", run, tag)
        continue
    !python -u scripts/run_m2.py {run} {extra} --tag {tag} 2>&1 | tee -a {DRIVE_ROOT}/runs/logs/m2_diag.log

## Bundle the results for Claude
This makes `MyDrive/pinn/m2_diag_results.zip` (small, no model files). Download it, put it in the `pinn` folder on the laptop, then tell Claude.

In [ ]:
out = f"{DRIVE_ROOT}/m2_diag_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob(f"{DRIVE_ROOT}/runs/**/*", recursive=True):
        rel = os.path.relpath(path, f"{DRIVE_ROOT}/runs")
        if os.path.isfile(path) and not path.endswith(".pt") and ("_diag" in rel or rel.endswith("m2_diag.log")):
            z.write(path, rel)
print("wrote", out, f"({os.path.getsize(out) / 1e6:.2f} MB)")